In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge


# =========================
# TRANSFORMATIONS
# =========================

def adstock_transform(x, decay):
    result = np.zeros_like(x, dtype=float)
    for t in range(len(x)):
        if t == 0:
            result[t] = x[t]
        else:
            result[t] = x[t] + decay * result[t - 1]
    return result


def hill_saturation(x, alpha, gamma):
    return (x ** alpha) / (x ** alpha + gamma ** alpha)


def transform_media(df, media_cols, adstock_decay, hill_alpha, hill_gamma):
    transformed = pd.DataFrame(index=df.index)
    
    for col in media_cols:
        adstocked = adstock_transform(df[col].values, adstock_decay[col])
        saturated = hill_saturation(adstocked, hill_alpha[col], hill_gamma[col])
        transformed[col] = saturated
        
    return transformed


# =========================
# PRIORS + SCALING
# =========================

def scale_media_for_roi(X, media_cols):
    X_scaled = X.copy()
    
    for col in media_cols:
        mean_val = X[col].mean()
        if mean_val != 0:
            X_scaled[col] = X[col] / mean_val
            
    return X_scaled


def build_feature_matrix(df, media_cols, non_media_cols,
                         adstock_decay, hill_alpha, hill_gamma):
    
    media_df = transform_media(df, media_cols,
                               adstock_decay, hill_alpha, hill_gamma)
    
    non_media_df = df[non_media_cols].copy()
    
    X = pd.concat([media_df, non_media_df], axis=1)
    
    return X


def build_prior_vector(feature_cols, media_cols,
                       roi_prior_mean, non_media_prior_mean):
    
    priors = []
    
    for col in feature_cols:
        if col in media_cols:
            priors.append(roi_prior_mean.get(col, 0))
        else:
            priors.append(non_media_prior_mean.get(col, 0))
    
    return np.array(priors)


# =========================
# MODEL TRAINING
# =========================

def train_mmm(df,
              target_col,
              media_cols,
              non_media_cols,
              adstock_decay,
              hill_alpha,
              hill_gamma,
              roi_prior_mean,
              non_media_prior_mean,
              ridge_alpha=1.0):
    
    # Build features
    X = build_feature_matrix(df,
                             media_cols,
                             non_media_cols,
                             adstock_decay,
                             hill_alpha,
                             hill_gamma)
    
    # Scale media for ROI interpretation
    X = scale_media_for_roi(X, media_cols)
    
    y = df[target_col].values
    feature_cols = X.columns.tolist()
    
    # Build prior
    prior = build_prior_vector(feature_cols,
                               media_cols,
                               roi_prior_mean,
                               non_media_prior_mean)
    
    # Fit ridge
    model = Ridge(alpha=ridge_alpha, fit_intercept=True)
    model.fit(X, y)
    
    return {
        "model": model,
        "feature_cols": feature_cols,
        "media_cols": media_cols,
        "non_media_cols": non_media_cols,
        "transform_params": {
            "adstock_decay": adstock_decay,
            "hill_alpha": hill_alpha,
            "hill_gamma": hill_gamma
        }
    }


# =========================
# PREDICTION
# =========================

def predict_mmm(model_obj, df):
    
    params = model_obj["transform_params"]
    
    X = build_feature_matrix(
        df,
        model_obj["media_cols"],
        model_obj["non_media_cols"],
        params["adstock_decay"],
        params["hill_alpha"],
        params["hill_gamma"]
    )
    
    X = scale_media_for_roi(X, model_obj["media_cols"])
    
    return model_obj["model"].predict(X)


# =========================
# CONTRIBUTIONS
# =========================

def get_contributions(model_obj, df):
    
    params = model_obj["transform_params"]
    
    X = build_feature_matrix(
        df,
        model_obj["media_cols"],
        model_obj["non_media_cols"],
        params["adstock_decay"],
        params["hill_alpha"],
        params["hill_gamma"]
    )
    
    X = scale_media_for_roi(X, model_obj["media_cols"])
    
    coefs = model_obj["model"].coef_
    
    contrib = X * coefs
    
    return contrib


# =========================
# ROI EXTRACTION
# =========================

def get_roi(model_obj):
    coefs = model_obj["model"].coef_
    return dict(zip(model_obj["feature_cols"], coefs))

In [ ]:
model = train_mmm(
    df=df,
    target_col="sales",
    media_cols=["tv", "youtube", "google"],
    non_media_cols=["price", "seasonality"],
    
    adstock_decay={"tv": 0.5, "youtube": 0.3, "google": 0.2},
    hill_alpha={"tv": 1.5, "youtube": 1.2, "google": 1.0},
    hill_gamma={"tv": 100, "youtube": 50, "google": 30},
    
    roi_prior_mean={"tv": 2.0, "youtube": 3.0, "google": 4.0},
    non_media_prior_mean={"price": -1.0, "seasonality": 0.5},
    
    ridge_alpha=10.0
)

preds = predict_mmm(model, df)

contrib = get_contributions(model, df)

roi = get_roi(model)